# Career26 Algorithmic Trading Strategy
### Mid-Frequency Mean-Reversion & Momentum Strategy

---

## 1. Competition Performance

I designed and deployed this algorithm in the **Career26 Algorithmic Trading Competition**, executing a multi-asset mean-reversion and momentum strategy across US large-cap equities.

* **Initial Capital:** $1,000,000.00
* **Total Realized Return:** **+5.08%**
* **Total Net P&L:** **+$51,062.00**
* **Maximum Drawdown (MDD):** **-1.31%**
* **Sharpe Ratio:** **0.02** The Sharpe ratio is very low since there was enormous volatility throughout much of the competition: part of it was mean to mimick the conditions of the 2008 financial crash.
* **Execution Interval:** 1.0s tick evaluation cadence
* **Universe (8 Symbols):** `AAPL`, `NVDA`, `GOOGL`, `TSLA`, `MSFT`, `AMZN`, `JPM`, `META`

---

## 2. Preliminary Definitions

* **Rolling Buffer ($B_{90}$):** The engine operates on high-frequency, tick-by-tick mid/last market quotes $P_t$ sampled at $1.0\text{ Hz}$. Let $B_{90}(t) = [P_{t-89}, P_{t-88}, \dots, P_t]$ denote the rolling 90-second price buffer. The 90-second fixed-capacity `deque(maxlen=90)` tracks rolling peak ($M$) and trough ($m$):
  $$M = \max(B_{90}), \quad m = \min(B_{90})$$
* **Short-Term Window ($B_{20}$):** Sub-slice evaluating the trailing 20 seconds of sampled tick data:
  $$B_{20} = [P_{t-19}, P_{t-18}, \dots, P_t]$$
* **Range Spread ($\Delta$):** Local absolute price bandwidth over the 90-second horizon:
  $$\Delta = M - m$$
* **Net Break-Even Price ($P_{\text{break\_even}}$):** Let $C_{\text{est}}$ be the estimated broker commission per trade, and $Q$ be the order volume. Entry cost adjusted for allocated two-way transaction costs:
  $$P_{\text{break\_even}} = P_{\text{entry}} + \frac{2C_{\text{est}}}{Q}$$

---

## 3. The Algorithm Execution Engine

### Phase 1: Entry Signal Vector (`FLAT` State)
To place an entry order, three conditions must evaluate to `True` simultaneously:

1. **Micro-Momentum Drift Filter:** Price must be non-decreasing ($P_i \ge P_{i-1}$) for at least 9 of the last 20 seconds:
   $$\sum_{i=1}^{19} \mathbb{I}_{\{P_{t-20+i} \ge P_{t-20+i-1}\}} \ge 9$$
   *Ensures the strategy buys into positive local drift rather than an accelerating downward trend.*

2. **Sub-Midpoint Price Location:** Current market price must reside in the lower half of the rolling 90-second band:
   $$m \le P_t \le m + 0.50\Delta$$

3. **Profitability Check:** Half the rolling spread must clear round-trip transaction costs:
   $$0.5\Delta \ge 2C_{\text{est}}$$

* **Entry Action:** Dispatches a limit buy order placed at the lower sextile:
  $$P_{\text{buy}} = m + \frac{\Delta}{6}$$
* **Limit Buy Timeout:** If unfilled after $90\text{ seconds}$, the engine cancels the order via REST API and resets status to `FLAT` to prevent fills in stale market regimes.

---

### Phase 2: Target & Risk Calibration (Upon Buy Fill)
When the broker confirms execution, the engine records $P_{\text{entry}} = \text{avg\_cost}$ and calculates three execution boundaries:

1. **Primary Profit Target (`limit_sell`):** Placed at $47\%$ of the range spread above the 90-second trough:
   $$P_{\text{target}} = m + 0.47\Delta$$
   *Stays below the structural midpoint ($0.47 < 0.50$) to avoid major liquidity resistance while securing high fill rates.*

2. **Defensive Recovery Target (`break_even_target`):** The lower value between a $0.10\%$ net gain and the primary target:
   $$P_{\text{recovery}} = \min\left(1.001P_{\text{entry}}+ \frac{2C_{\text{est}}}{Q}, P_{\text{target}}\right)$$

3. **Hard Stop Loss Trigger (`stop_loss_price`):** Placed strictly $0.4\%$ below entry cost:
   $$P_{\text{stop\_trigger}} = 0.996P_{\text{entry}}$$

The engine immediately issues a limit sell order at $P_{\text{target}}$ and transitions to `HOLDING_TARGET_LIMIT`.

---

### Phase 3: Position Management (`HOLDING_TARGET_LIMIT`)
While holding inventory, the engine evaluates three dynamic state transitions every second:

* **Hard Stop Loss Execution:**
  * **Trigger:** $P_t \le P_{\text{stop\_trigger}}$ (drawdown reaches $-0.4\%$).
  * **Action:** Cancels open target limits and places a defensive limit sell at $-0.43\%$ below entry:
    $$P_{\text{stop\_sell}} = 0.9957P_{\text{entry}}$$
    *Transitions state to `PENDING_SELL` with buffer for downward continuation.*

* **25-Second Profit-Preservation Sweep:**
  * **Trigger:** $t - t_{\text{holding}} > 25\text{s}$ **AND** $P_t \ge P_{\text{break\_even}}$.
  * **Action:** Cancels the primary target limit and places a limit sell at current market price ($P_t$) to lock in a non-negative ($\ge \$0.00$) realized P&L.

* **Adverse Drift Trigger (Shift to Recovery):**
  * **Trigger:** Price trades below entry cost for $\ge 16$ of the trailing 20 seconds:
    $$\sum_{i=0}^{19} \mathbb{I}_{\{P_{t-i} < P_{\text{entry}}\}} \ge 16$$
  * **Action:** Cancels the primary target limit and shifts state to `RECOVERY_EXIT_MODE`.

---

### Phase 4: Defensive Exits (`RECOVERY_EXIT_MODE`)

1. **Standard Recovery Target Hit:**
   * When $P_t \ge P_{\text{recovery}}$, submits a limit sell order at $P_{\text{recovery}}$.
2. **25-Second Profit-Preservation Sweep:**
   * **Trigger:** $t - t_{\text{recovery}} > 25\text{s}$ **AND** $P_t \ge P_{\text{break\_even}}$.
   * **Action:** Cancels the primary target limit and places a limit sell at current market price ($P_t$) to lock in a non-negative ($\ge \$0.00$) realized P&L.
---

### Phase 5: Safety Nets, Server Auditing & Dashboard

* **`PENDING_SELL` Timeout Safety Net:** If any limit sell hangs in `PENDING_SELL` for $> 45\text{ seconds}$ without filling, the engine cancels it and issues a market order (`type="market"`) to force-close inventory.
* **Global Server Reconciliation:** Every cycle, if server reports `actual_qty == 0` for any non-flat symbol, the trade is confirmed closed. Net P&L (after round-trip fees) is logged to `trade_history` and state fully resets to `FLAT`.
* **Live Dashboard:** Terminal view clears and updates every second, displaying the **Active Positions & Order States** table at the bottom beneath the **Fill Audit Log** and **Completed Trades P&L** summaries.
---

## 4. Summary of Order Change Rules

The table below outlines every deterministic trigger where limit orders are cancelled, amended, or converted into market orders:

| State Transition | Trigger Condition | Engine Action | New Order / Target Price | Rationale |
| :--- | :--- | :--- | :--- | :--- |
| **`PENDING_BUY` $\to$ `FLAT`** | $t - t_{\text{order}} > 90\text{s}$ and $Q_{\text{actual}} = 0$ | `DELETE /orders/{sym}` | *None (Reset to FLAT)* | **Entry Expiry:** Stale order cancellation; market regime shifted away from entry level. |
| **`PENDING_BUY` $\to$ `HOLDING_TARGET_LIMIT`** | $Q_{\text{actual}} > 0$ (Fill confirmed) | `POST /orders` (Sell Limit) | $P_{\text{target}} = m + 0.47\Delta$ | **Initial Limit Target:** Placed immediately to capture sub-midpoint mean reversion. |
| **`HOLDING_TARGET_LIMIT` $\to$ `PENDING_SELL`**<br>*(Primary 25s Timeout)* | $t - t_{\text{holding}} > 25\text{s}$<br>**AND** $P_t \ge P_{\text{entry}} + \frac{2C_{\text{est}}}{Q}$ | `DELETE /orders/{sym}`<br>`POST /orders` (Sell Limit) | **$P_{\text{limit\_sell}} = P_t$ (Current Market Price)** | **Alpha Sweep:** If held $> 25\text{s}$ and trade is in net green, lock in profit immediately at current bid/mid. |
| **`HOLDING_TARGET_LIMIT` $\to$ `RECOVERY_EXIT_MODE`** | $\sum_{i=0}^{19} \mathbb{I}_{\{P_{t-i} < P_{\text{entry}}\}} \ge 16$ | `DELETE /orders/{sym}` | $P_{\text{recovery}} = \min(P_{\text{target}}, P_{\text{entry}} \times 1.001 + \frac{2C_{\text{est}}}{Q})$ | **Adverse Drift Switch:** Price under water for $\ge 80\%$ of last 20s; abandon optimistic target. |
| **`RECOVERY_EXIT_MODE` $\to$ `PENDING_SELL`**<br>*(Standard Recovery Hit)* | $P_t \ge P_{\text{recovery}}$ | `POST /orders` (Sell Limit) | $P_{\text{limit\_sell}} = P_{\text{recovery}}$ | **Target Execution:** Exit trade at defensive target upon first upward tick rebound. |
| **`RECOVERY_EXIT_MODE` $\to$ `PENDING_SELL`**<br>*(Recovery 25s Timeout)* | $t - t_{\text{recovery}} > 25\text{s}$<br>**AND** $P_t \ge P_{\text{entry}} + \frac{2C_{\text{est}}}{Q}$ | `DELETE /orders/{sym}`<br>`POST /orders` (Sell Limit) | **$P_{\text{limit\_sell}} = P_t$ (Current Market Price)** | **Defensive Liquidation:** Scrap recovery limit and cross spread at market to free capital if trade breaks even. |
| **`HOLDING_TARGET_LIMIT` $\to$ `PENDING_SELL`**<br>*(Hard Stop Loss Trigger)* | $P_t \le 0.996P_{\text{entry}}$ | `DELETE /orders/{sym}`<br>`POST /orders` (Stop Limit) | $P_{\text{stop\_sell}} = 0.9957P_{\text{entry}}$ | **Catastrophic Tail Risk Protection:** Places limit sell $3\text{ bps}$ below stop trigger to balance fill rate vs slippage. |
| **`PENDING_SELL` $\to$ Closed**<br>*(Emergency Fail-Safe)* | $t - t_{\text{sell\_order}} > 45\text{s}$<br>**AND** $Q_{\text{actual}} > 0$ | `DELETE /orders/{sym}`<br>`POST /orders` (Market Sell) | **Market Order (`type="market"`)** | **Execution Guarantee:** Flushes hanging limit sells that missed the order book queue. |

We will now provide the actual code used for execution.

---

In [ ]:
import time
import zoneinfo
from datetime import datetime
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from IPython.display import display, clear_output

# =========================================================
# CONFIGURATION
# =========================================================
SYMBOLS = [
    # Core Tech & Growth
    "AAPL", "NVDA", "GOOGL", "TSLA", "MSFT",
    # Additional Backend-Supported Single-Name Equities
    "AMZN", "JPM", "META"
]
ORDER_QTY = 60                       # 60 shares per trade
ESTIMATED_COMMISSION_PER_TRADE = 0.0  # Set to your fee per trade if non-zero
LIMIT_BUY_TIMEOUT_SECONDS = 90        # 90-second limit buy order timeout
MIN_NON_DECREASING_SECS = 9           # Entry momentum 9/20s non-decreasing
MIN_BELOW_COST_SECS = 16              # Recovery mode trigger: below cost >= 16/20s
TARGET_PROFIT_PCT = 0.001             # 0.1% profit target in recovery mode
HARD_STOP_LOSS_PCT = 0.004            # Hard Stop Trigger floor (0.4% below purchase cost)
STOP_LOSS_SELL_PCT = 0.0043           # Stop Loss Limit Sell placed at 0.43% below entry cost
PROFIT_ADJUST_TIMEOUT_SECS = 25       # 25-second timeout to adjust sell limit if net profit >= 0

# Safe state tracker per symbol
trade_state = {
    sym: {
        "status": "FLAT",            # "FLAT", "PENDING_BUY", "HOLDING_TARGET_LIMIT", "RECOVERY_EXIT_MODE", "PENDING_SELL"
        "entry_price": 0.0,
        "qty": 0,
        "order_timestamp": None,
        "holding_start_time": None,   # Timestamp when buy filled
        "recovery_start_time": None,  # Timestamp when recovery mode started
        "limit_buy_price": 0.0,
        "target_sell_price": 0.0,
        "break_even_target": 0.0,
        "stop_loss_price": 0.0
    } for sym in SYMBOLS
}

# Audit Logs
orders_fill_log = []  # Complete log of ALL filled buy & sell orders
trade_history = []    # Summary log of COMPLETED round-trip trades with P&L

def log_order_fill(symbol: str, side: str, qty: float, price: float, order_type: str, status_note: str, timestamp_str: str, pnl_str: str = "-"):
    """Appends an execution record to the permanent All Orders & Fills Log with timestamp and sell P&L."""
    orders_fill_log.append({
        "Execution Time": timestamp_str,
        "Symbol": symbol,
        "Side": side.upper(),
        "Qty": qty,
        "Fill Price": f"${price:.2f}",
        "Order Type": order_type.upper(),
        "Execution Note": status_note,
        "Realized PnL": pnl_str
    })

def export_logs_to_csv():
    """Exports both audit logs to CSV files on disk at session end."""
    print("\n" + "="*60)
    print(" EXPORTING STRATEGY AUDIT LOGS TO CSV...")
    print("="*60)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # 1. Export All Orders & Fills Audit Log
    if orders_fill_log:
        df_orders = pd.DataFrame(orders_fill_log)
        orders_filename = f"audit_orders_fill_log_{timestamp}.csv"
        df_orders.to_csv(orders_filename, index=False)
        print(f" [SUCCESS] Saved {len(df_orders)} order events -> '{orders_filename}'")
    else:
        print(" [INFO] No orders were filled during this session.")

    # 2. Export Completed Trades & Realized P&L Log
    if trade_history:
        df_trades = pd.DataFrame(trade_history)
        trades_filename = f"completed_trades_pnl_log_{timestamp}.csv"
        df_trades.to_csv(trades_filename, index=False)
        print(f" [SUCCESS] Saved {len(df_trades)} completed trades -> '{trades_filename}'")
    else:
        print(" [INFO] No round-trip trades were completed during this session.")

    print("="*60 + "\n")

def fetch_network_data_parallel(symbols: list):
    """Executes REST API queries concurrently in parallel threads."""
    with ThreadPoolExecutor(max_workers=len(symbols) + 1) as executor:
        account_future = executor.submit(get_account)
        quote_futures = {sym: executor.submit(get_quote, sym) for sym in symbols}

        try:
            account_data = account_future.result()
            positions = account_data.get("positions", {})
        except Exception:
            positions = {}

        quotes = {}
        for sym, fut in quote_futures.items():
            try:
                quotes[sym] = fut.result()
            except Exception:
                quotes[sym] = {}

    return positions, quotes

def run_strategy(symbols: list, interval_seconds: float = 1.0):
    uk_tz = zoneinfo.ZoneInfo("Europe/London")
    price_buffers = {sym: deque(maxlen=90) for sym in symbols}

    print(f"Starting Engine (9/20s Momentum | Target m+0.47(M-m)) for {symbols}...")
    time.sleep(1)

    while True:
        now_uk = datetime.now(uk_tz)
        time_str = now_uk.strftime('%H:%M:%S')

        # ---------------------------------------------------------
        # 5:00 PM UK STOP CONDITION & AUTO CSV EXPORT
        # ---------------------------------------------------------
        if now_uk.hour >= 17:
            print(f"\n[{time_str} UK] Market session closed. Shutting down trading engine.")
            export_logs_to_csv()
            break

        start_tick = time.perf_counter()
        dashboard_status = {}

        # Parallel Network Fetching
        positions, quotes = fetch_network_data_parallel(symbols)

        for symbol in symbols:
            try:
                quote = quotes.get(symbol, {})
                current_price = float(quote.get("last") or quote.get("mid") or quote.get("ask"))

                # Update 90s rolling buffer
                buf = price_buffers[symbol]
                buf.append(current_price)

                state = trade_state[symbol]

                # Query server position state
                server_pos = positions.get(symbol, {})
                actual_qty = server_pos.get("qty", 0)
                avg_cost = server_pos.get("avg_cost", state["entry_price"])

                # =========================================================
                # GLOBAL POSITION AUDIT: FORCE RESET IF POSITION IS CLOSED
                # =========================================================
                if actual_qty == 0 and state["status"] in ["HOLDING_TARGET_LIMIT", "RECOVERY_EXIT_MODE", "PENDING_SELL"]:
                    exit_price = current_price
                    qty_sold = state["qty"] if state["qty"] > 0 else ORDER_QTY
                    
                    # Net P&L after round-trip commissions
                    pnl = ((exit_price - state["entry_price"]) * qty_sold) - (2 * ESTIMATED_COMMISSION_PER_TRADE)
                    pnl_pct = ((exit_price / state["entry_price"]) - 1) * 100 if state["entry_price"] > 0 else 0
                    pnl_formatted = f"${pnl:+.2f} ({pnl_pct:+.2f}%)"

                    log_order_fill(symbol, "SELL", qty_sold, exit_price, "LIMIT", "SELL FILLED CONFIRMED", time_str, pnl_str=pnl_formatted)

                    trade_history.append({
                        "Close Time": time_str,
                        "Symbol": symbol,
                        "Qty": qty_sold,
                        "Buy Price": f"${state['entry_price']:.2f}",
                        "Sell Price": f"${exit_price:.2f}",
                        "P&L ($)": round(pnl, 2),
                        "P&L (%)": f"{pnl_pct:+.2f}%"
                    })
                    print(f"\n[{time_str}] [POSITION CLOSED CONFIRMED] {symbol} @ ${exit_price:.2f} | Realized PnL: {pnl_formatted}")

                    # Reset symbol state completely to FLAT
                    state["status"] = "FLAT"
                    state["entry_price"] = 0.0
                    state["qty"] = 0
                    state["order_timestamp"] = None
                    state["holding_start_time"] = None
                    state["recovery_start_time"] = None
                    state["stop_loss_price"] = 0.0
                    state["break_even_target"] = 0.0
                    state["limit_buy_price"] = 0.0
                    state["target_sell_price"] = 0.0

                # ---------------------------------------------------------
                # STATE MACHINE EVALUATION
                # ---------------------------------------------------------

                # CASE 1: Waiting for Limit Buy to fill
                if state["status"] == "PENDING_BUY":
                    elapsed_pending = time.time() - (state["order_timestamp"] or time.time())

                    if actual_qty > 0:
                        # Buy Order Filled! Calculate targets
                        M = max(buf)
                        m = min(buf)
                        range_spread = M - m

                        # PRIMARY TARGET: m + 0.47 * (M - m)
                        limit_sell = round(m + (0.47 * range_spread), 2)
                        total_commissions = 2 * ESTIMATED_COMMISSION_PER_TRADE
                        per_share_commission = total_commissions / actual_qty

                        profit_target_price = round((avg_cost * (1 + TARGET_PROFIT_PCT)) + per_share_commission, 2)
                        break_even_target = min(profit_target_price, limit_sell)
                        stop_loss_price = round(avg_cost * (1 - HARD_STOP_LOSS_PCT), 2)

                        state["entry_price"] = avg_cost
                        state["qty"] = actual_qty
                        state["target_sell_price"] = limit_sell
                        state["break_even_target"] = break_even_target
                        state["stop_loss_price"] = stop_loss_price
                        state["holding_start_time"] = time.time()

                        log_order_fill(symbol, "BUY", actual_qty, avg_cost, "LIMIT", "ENTRY LIMIT FILLED", time_str, pnl_str="-")

                        print(f"\n[{time_str}] [BUY EXECUTION CONFIRMED] BOUGHT {actual_qty} {symbol} @ ${avg_cost:.2f}.")
                        print(f"Target Sell @ ${limit_sell:.2f} | Recovery Target @ ${break_even_target:.2f} | Hard Stop Trigger (0.4%) @ ${stop_loss_price:.2f}")

                        sell(symbol, qty=actual_qty, order_type="limit", limit_price=limit_sell)
                        state["status"] = "HOLDING_TARGET_LIMIT"

                    elif elapsed_pending > LIMIT_BUY_TIMEOUT_SECONDS:
                        print(f"\n[{time_str}] [TIMEOUT] Limit Buy for {symbol} unfilled after {int(elapsed_pending)}s. Cancelling order & resetting...")
                        try:
                            requests.delete(f"{BACKEND}/orders/{symbol}", headers=HEADERS)
                        except Exception:
                            pass
                        state["status"] = "FLAT"
                        state["entry_price"] = 0.0
                        state["qty"] = 0

                # CASE 2: Holding Position - Monitor Dynamic Exits
                elif state["status"] in ["HOLDING_TARGET_LIMIT", "RECOVERY_EXIT_MODE"]:

                    per_share_comm = (2 * ESTIMATED_COMMISSION_PER_TRADE) / actual_qty if actual_qty > 0 else 0
                    min_break_even_price = state["entry_price"] + per_share_comm

                    # --- A. HARD STOP LOSS CHECK (Placing Limit Sell @ 0.43% below entry) ---
                    if current_price <= state["stop_loss_price"] and actual_qty > 0:
                        stop_sell_limit_price = round(state["entry_price"] * (1 - STOP_LOSS_SELL_PCT), 2)
                        pnl = ((stop_sell_limit_price - state["entry_price"]) * actual_qty) - (2 * ESTIMATED_COMMISSION_PER_TRADE)
                        pnl_pct = ((stop_sell_limit_price / state["entry_price"]) - 1) * 100
                        pnl_formatted = f"${pnl:+.2f} ({pnl_pct:+.2f}%)"

                        print(f"\n[{time_str}] [STOP LOSS TRIGGERED - {symbol}] Current Price ${current_price:.2f} <= Stop Trigger ${state['stop_loss_price']:.2f}")
                        print(f"Cancelling target limit orders and placing STOP LIMIT SELL @ ${stop_sell_limit_price:.2f} (-0.43%).")

                        try:
                            requests.delete(f"{BACKEND}/orders/{symbol}", headers=HEADERS)
                        except Exception:
                            pass

                        sell(symbol, qty=actual_qty, order_type="limit", limit_price=stop_sell_limit_price)
                        log_order_fill(symbol, "SELL", actual_qty, stop_sell_limit_price, "LIMIT", "STOP LOSS LIMIT ORDER PLACED (-0.43%)", time_str, pnl_str=pnl_formatted)

                        state["status"] = "PENDING_SELL"
                        state["order_timestamp"] = time.time()

                    # --- B. 25-SECOND RULE FOR PRIMARY TARGET (HOLDING_TARGET_LIMIT) ---
                    elif state["status"] == "HOLDING_TARGET_LIMIT" and state["holding_start_time"] and (time.time() - state["holding_start_time"]) > PROFIT_ADJUST_TIMEOUT_SECS and current_price >= min_break_even_price and actual_qty > 0:
                        print(f"\n[{time_str}] [25s TIMEOUT - PRIMARY TARGET ADJUST] {symbol} holding > 25s. Current price ${current_price:.2f} >= break-even ${min_break_even_price:.2f}.")
                        print(f"Scrapping original limit target (${state['target_sell_price']:.2f}) and placing limit sell @ market price (${current_price:.2f}).")

                        try:
                            requests.delete(f"{BACKEND}/orders/{symbol}", headers=HEADERS)
                        except Exception:
                            pass

                        sell(symbol, qty=actual_qty, order_type="limit", limit_price=current_price)
                        state["status"] = "PENDING_SELL"
                        state["order_timestamp"] = time.time()

                    # --- C. CHECK RECOVERY EXIT TRIGGER (16 of last 20s below cost) ---
                    elif state["status"] == "HOLDING_TARGET_LIMIT" and len(buf) >= 20:
                        buf_tuple = tuple(buf)
                        last_20 = buf_tuple[-20:]
                        secs_below_cost = sum(1 for p in last_20 if p < state["entry_price"])

                        if secs_below_cost >= MIN_BELOW_COST_SECS:
                            print(f"\n[{time_str}] [SCRAPPING LIMIT ORDER - {symbol}] Price below cost for {secs_below_cost}/20s.")
                            print(f"Switching to RECOVERY EXIT MODE (Targeting min(0.1% profit, orig limit) = ${state['break_even_target']:.2f})")

                            try:
                                requests.delete(f"{BACKEND}/orders/{symbol}", headers=HEADERS)
                            except Exception:
                                pass

                            state["status"] = "RECOVERY_EXIT_MODE"
                            state["recovery_start_time"] = time.time()

                    # --- D. RECOVERY EXIT EXECUTION ---
                    elif state["status"] == "RECOVERY_EXIT_MODE" and actual_qty > 0:
                        elapsed_recovery = time.time() - (state["recovery_start_time"] or time.time())

                        # Standard recovery execution target hit
                        if current_price >= state["break_even_target"]:
                            pnl = ((current_price - state["entry_price"]) * actual_qty) - (2 * ESTIMATED_COMMISSION_PER_TRADE)
                            pnl_pct = ((current_price / state["entry_price"]) - 1) * 100
                            pnl_formatted = f"${pnl:+.2f} ({pnl_pct:+.2f}%)"

                            print(f"\n[{time_str}] [RECOVERY EXIT EXECUTED - {symbol}] Price ${current_price:.2f} >= Target ${state['break_even_target']:.2f} | Realized PnL: {pnl_formatted}")

                            sell(symbol, qty=actual_qty, order_type="limit", limit_price=state["break_even_target"])
                            log_order_fill(symbol, "SELL", actual_qty, current_price, "LIMIT", "RECOVERY EXIT FILLED", time_str, pnl_str=pnl_formatted)

                            state["status"] = "PENDING_SELL"
                            state["order_timestamp"] = time.time()

                        # 25-second Rule in Recovery Mode (If >= 25s and net profit >= 0)
                        elif elapsed_recovery > PROFIT_ADJUST_TIMEOUT_SECS and current_price >= min_break_even_price:
                            print(f"\n[{time_str}] [25s TIMEOUT - RECOVERY ADJUST] {symbol} in recovery mode > 25s. Current price ${current_price:.2f} >= break-even ${min_break_even_price:.2f}.")
                            print(f"Scrapping recovery target (${state['break_even_target']:.2f}) and placing limit sell @ market price (${current_price:.2f}).")

                            try:
                                requests.delete(f"{BACKEND}/orders/{symbol}", headers=HEADERS)
                            except Exception:
                                pass

                            sell(symbol, qty=actual_qty, order_type="limit", limit_price=current_price)
                            state["status"] = "PENDING_SELL"
                            state["order_timestamp"] = time.time()

                # CASE 3: PENDING_SELL TIMEOUT SAFETY NET
                elif state["status"] == "PENDING_SELL":
                    elapsed_sell = time.time() - (state["order_timestamp"] or time.time())
                    # If pending sell hangs for more than 45 seconds without filling, exit via Market Order
                    if elapsed_sell > 45 and actual_qty > 0:
                        print(f"\n[{time_str}] [PENDING SELL TIMEOUT - {symbol}] Limit sell unfilled after 45s. Market selling to force-close position...")
                        try:
                            requests.delete(f"{BACKEND}/orders/{symbol}", headers=HEADERS)
                        except Exception:
                            pass
                        sell(symbol, qty=actual_qty, order_type="market")

                # ---------------------------------------------------------
                # STRATEGY ENTRY EVALUATION (Only when FLAT)
                # ---------------------------------------------------------
                if state["status"] == "FLAT" and len(buf) >= 20:
                    M = max(buf)
                    m = min(buf)
                    range_spread = M - m

                    buf_tuple = tuple(buf)
                    last_20 = buf_tuple[-20:]
                    non_decreasing = sum(1 for i in range(1, len(last_20)) if last_20[i] >= last_20[i-1])

                    cond_a = non_decreasing >= MIN_NON_DECREASING_SECS
                    cond_b = m <= current_price <= (m + range_spread / 2.0)
                    cond_step3 = (0.5 * range_spread) >= (2 * ESTIMATED_COMMISSION_PER_TRADE)

                    if cond_a and cond_b and cond_step3:
                        limit_buy = round(m + (range_spread / 6.0), 2)

                        buy(symbol, qty=ORDER_QTY, order_type="limit", limit_price=limit_buy)

                        state["status"] = "PENDING_BUY"
                        state["entry_price"] = limit_buy
                        state["qty"] = ORDER_QTY
                        state["order_timestamp"] = time.time()
                        state["limit_buy_price"] = limit_buy

                        print(f"\n[{time_str}] [BUY ORDER SENT] Limit Buy {ORDER_QTY} {symbol} @ ${limit_buy:.2f}. Timeout timer started (90s)...")

                # Dashboard formatting
                if state["status"] == "PENDING_BUY":
                    time_left = max(0, int(LIMIT_BUY_TIMEOUT_SECONDS - (time.time() - (state["order_timestamp"] or time.time()))))
                    status_text = f"PENDING_BUY (Timeout in {time_left}s)"
                else:
                    status_text = state["status"]

                dashboard_status[symbol] = {
                    "Price": f"${current_price:.2f}",
                    "State": status_text,
                    "Entry": f"${state['entry_price']:.2f}" if state['entry_price'] > 0 else "-",
                    "Hard Stop (0.4%)": f"${state['stop_loss_price']:.2f}" if state['stop_loss_price'] > 0 else "-",
                    "Recovery Target": f"${state['break_even_target']:.2f}" if state['break_even_target'] > 0 else "-",
                    "Unrealized P&L": f"${((current_price - state['entry_price']) * actual_qty) - (2 * ESTIMATED_COMMISSION_PER_TRADE):+.2f}" if "HOLDING" in state["status"] or state["status"] == "RECOVERY_EXIT_MODE" else "$0.00"
                }

            except Exception as e:
                dashboard_status[symbol] = {"Error": str(e)}

        # ---------------------------------------------------------
        # DASHBOARD DISPLAY REFRESH (POSITIONS BOX AT THE BOTTOM)
        # ---------------------------------------------------------
        clear_output(wait=True)
        print(f"=== STRATEGY ENGINE (9/20s MOMENTUM | TARGET m+0.47(M-m)) | UK Time: {now_uk.strftime('%H:%M:%S')} ===")

        if orders_fill_log:
            print("\n--- ALL ORDERS & FILLED PRICES AUDIT LOG (WITH TIME & PnL) ---")
            display(pd.DataFrame(orders_fill_log))

        if trade_history:
            print("\n--- COMPLETED TRADES & REALIZED P&L ---")
            display(pd.DataFrame(trade_history))
            total_realized = sum(t["P&L ($)"] for t in trade_history)
            print(f"TOTAL REALIZED P&L: ${total_realized:+.2f}")

        print("\n--- ACTIVE POSITIONS & ORDER STATES ---")
        display(pd.DataFrame(dashboard_status).T)

        # Precision cadence lock
        elapsed = time.perf_counter() - start_tick
        time.sleep(max(0.0, interval_seconds - elapsed))

# Run engine
run_strategy(SYMBOLS, interval_seconds=1.0)